# Simulation Playground

Run all three assignment strategies (A: Uniform, B: Load-Minimising, C: Load-Maximising) against a shared warehouse and compare outcomes interactively.

**Workflow:**
1. Edit parameters in **Cell 2**.
2. Run all cells top-to-bottom (`Kernel → Restart & Run All`).
3. Re-run individual sections after tweaking a single parameter group.


In [ ]:
# ── Cell 1 · Imports & path setup ─────────────────────────────────────────────
import math, os, random, sys, time
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')   # headless; swap to 'inline' if running Jupyter locally
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from IPython.display import display

_HERE = os.path.abspath('')          # notebook dir = Optimization/
_REPO = os.path.normpath(os.path.join(_HERE, '..'))
sys.path.insert(0, os.path.join(_REPO, 'Warehouse'))
sys.path.insert(0, _HERE)

from Aisle_Storage import Aisle
from Affinity_Store import AffinityStore
from Inventory_Management import (
    Inventory_Manager, LoadParams,
    build_load_minimizing_assignment_fn,
    build_load_maximizing_assignment_fn,
)
from generate_inventory import load_inventory_from_db
from Pick import PickConfig, PickSimulation
from Warehouse_Builder import AisleConfig, Warehouse_Builder, WarehouseConfig
from Workload_Builder import Batch, BatchConfig, Task
from Simulation_Analytics import extract_batch_stats, extract_task_stats
from Workload import WorkloadParams

print('Imports OK')

In [ ]:
# ── Cell 2 · PARAMETERS ── edit everything here ───────────────────────────────

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED_WORLD   = 42      # warehouse layout + initial stocking
SEED_BATCHES = 1337    # batch generation

# ── Simulation volume ─────────────────────────────────────────────────────────
N_BATCHES  = 200       # batches to simulate
K_PICKERS  = 25        # pickers per batch

# ── Pick model ────────────────────────────────────────────────────────────────
# W_a = travel + Σ(intercept + weight_coef·ln(w)·qty + volume_coef·ln(v)·qty)
#       + cart_swap_coef · (extra_carts)
PICK_X_MOVE_TIME    = 1.0     # time units per bayX step
PICK_Y_MOVE_TIME    = 0.5     # time units per bayY step
PICK_INTERCEPT      = 1.0     # fixed overhead per pick stop
PICK_WEIGHT_COEF    = 1.1     # weight sensitivity
PICK_VOLUME_COEF    = 1e-3    # volume sensitivity
CART_SWAP_COEF      = 10.0    # penalty per extra cart

# ── Load model ────────────────────────────────────────────────────────────────
# L_a = W_a + λ·(W_a/k)^γ · lift_sum
LOAD_LAMBDA = 1.1     # startup-cost multiplier
LOAD_K      = 1.0     # pickers per task (normally 1 for single-aisle)
LOAD_GAMMA  = 1.5     # congestion exponent

# ── Batch generation ──────────────────────────────────────────────────────────
BATCH_MEAN_FRAC = 0.25    # centre of #SKUs-per-batch distribution
BATCH_STD_FRAC  = 0.03    # spread

# ── Warehouse sizing ──────────────────────────────────────────────────────────
SING_FRACTION_CAP = 0.35   # singleton bins ≤ 35% of total warehouse bins
TARGET_FILL       = 0.90   # target bin fill rate after overstock sampling

# ── Data ──────────────────────────────────────────────────────────────────────
# Set INVENTORY_DB / AFFINITY_DB to explicit paths, or leave as None to
# auto-detect the latest generated batch.
INVENTORY_DB: str | None = None
AFFINITY_DB:  str | None = None

# ── Rolling-mean window for timeseries plots ──────────────────────────────────
PLOT_WINDOW = 30

print('Parameters set.')

In [ ]:
# ── Cell 3 · Resolve DB paths ─────────────────────────────────────────────────
_BATCHES_DIR = os.path.join(_REPO, 'Warehouse', 'generated', 'batches')

def _find_latest_pair(batches_dir):
    if not os.path.isdir(batches_dir):
        return None, None
    for batch in reversed(sorted(os.listdir(batches_dir))):
        bp = os.path.join(batches_dir, batch)
        if not os.path.isdir(bp):
            continue
        for profile in sorted(os.listdir(bp)):
            pp = os.path.join(bp, profile)
            i  = os.path.join(pp, 'inventory', 'inventory.db')
            a  = os.path.join(pp, 'affinity',  'affinity.db')
            if os.path.exists(i) and os.path.exists(a):
                return i, a
    return None, None

_inv_db  = INVENTORY_DB or _find_latest_pair(_BATCHES_DIR)[0]
_aff_db  = AFFINITY_DB  or _find_latest_pair(_BATCHES_DIR)[1]

assert _inv_db and os.path.exists(_inv_db), f'inventory.db not found: {_inv_db}'
assert _aff_db and os.path.exists(_aff_db), f'affinity.db not found: {_aff_db}'

print(f'Inventory : {_inv_db}')
print(f'Affinity  : {_aff_db}')

In [ ]:
# ── Cell 4 · Load inventory + affinity ────────────────────────────────────────
t0        = time.perf_counter()
inventory = load_inventory_from_db(_inv_db)
n_skus    = len(inventory.cartons)
print(f'{n_skus:,} SKUs loaded  ({time.perf_counter()-t0:.2f}s)')

t0             = time.perf_counter()
affinity_store = AffinityStore(_aff_db)
n_aff          = affinity_store._matrix.nnz if affinity_store._matrix is not None else 0
mb             = 0 if affinity_store._matrix is None else (
    affinity_store._matrix.data.nbytes +
    affinity_store._matrix.indices.nbytes +
    affinity_store._matrix.indptr.nbytes) / 1_048_576
print(f'Affinity CSR: {n_aff:,} entries  {mb:.0f} MB  ({time.perf_counter()-t0:.1f}s)')

# Derived config objects
pick_cfg   = PickConfig(
    num_pickers      = K_PICKERS,
    x_move_time      = PICK_X_MOVE_TIME,
    y_move_time      = PICK_Y_MOVE_TIME,
    pick_intercept   = PICK_INTERCEPT,
    pick_weight_coef = PICK_WEIGHT_COEF,
    pick_volume_coef = PICK_VOLUME_COEF,
    cart_swap_coef   = CART_SWAP_COEF,
)
wp          = WorkloadParams.from_pick_config(pick_cfg)
load_params = LoadParams(lambda_=LOAD_LAMBDA, k=LOAD_K, gamma=LOAD_GAMMA)
batch_cfg   = BatchConfig(
    inventory_size = n_skus,
    mean_fraction  = BATCH_MEAN_FRAC,
    std_fraction   = BATCH_STD_FRAC,
)
print('Config objects built.')

In [ ]:
# ── Cell 5 · Warehouse layout constants ───────────────────────────────────────
_BINS_PER_AISLE    = 25 * 20
_N_PALLET_TYPES    = 48
_N_SINGLETON_TYPES = 12
_SINGLETON_MAX_DIM = 16

_CATEGORIES  = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
_ALL_SIZES   = ['small', 'medium', 'large', 'extra_large']
_CONV_SIZES  = ['small', 'medium', 'large']
_CONV_PROBS  = [0.25, 0.50, 0.25]
_NCONV_SIZES = ['medium', 'large', 'extra_large']
_NCONV_PROBS = [0.20, 0.50, 0.30]

_AISLE_CFGS = []
for _sz in _ALL_SIZES:
    for _cat in _CATEGORIES:
        _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'pallet', 25, 20, [_sz],        None))
        _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'pallet', 25, 20, [_sz],        None))
for _cat in _CATEGORIES:
    _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'singleton', 25, 20, _CONV_SIZES,  _CONV_PROBS))
    _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'singleton', 25, 20, _NCONV_SIZES, _NCONV_PROBS))

n_singleton   = sum(1 for c in inventory.cartons if max(c.length, c.width, c.height) <= _SINGLETON_MAX_DIM)
n_pallet      = n_skus - n_singleton
sing_replicas = max(1, math.ceil(n_singleton / (_N_SINGLETON_TYPES * _BINS_PER_AISLE)))
pall_replicas = max(1, math.ceil(n_pallet    / (_N_PALLET_TYPES    * _BINS_PER_AISLE)))

sing_bins_now = _N_SINGLETON_TYPES * sing_replicas * _BINS_PER_AISLE
min_pall_bins = math.ceil(sing_bins_now * (1 - SING_FRACTION_CAP) / SING_FRACTION_CAP)
pall_replicas = max(pall_replicas, math.ceil(min_pall_bins / (_N_PALLET_TYPES * _BINS_PER_AISLE)))

sing_bins  = _N_SINGLETON_TYPES * sing_replicas * _BINS_PER_AISLE
pall_bins  = _N_PALLET_TYPES    * pall_replicas * _BINS_PER_AISLE
min_total  = math.ceil(n_skus / TARGET_FILL)
if sing_bins + pall_bins < min_total:
    extra = math.ceil((min_total - sing_bins - pall_bins) / (_N_PALLET_TYPES * _BINS_PER_AISLE))
    pall_replicas += extra

total_aisles = _N_SINGLETON_TYPES * sing_replicas + _N_PALLET_TYPES * pall_replicas
total_bins   = total_aisles * _BINS_PER_AISLE
sing_frac    = (_N_SINGLETON_TYPES * sing_replicas) / total_aisles
aisle_splits = [pall_replicas / total_aisles] * _N_PALLET_TYPES + \
               [sing_replicas / total_aisles] * _N_SINGLETON_TYPES

warehouse_cfg = WarehouseConfig(
    total_aisles  = total_aisles,
    aisle_splits  = aisle_splits,
    aisle_configs = _AISLE_CFGS,
)

print(f'Inventory  : {n_pallet:,} pallet  {n_singleton:,} singleton cartons')
print(f'Warehouse  : {_N_PALLET_TYPES}×{pall_replicas} pallet  +  {_N_SINGLETON_TYPES}×{sing_replicas} singleton')
print(f'           : {total_aisles} aisles  /  {total_bins:,} bins  sing={sing_frac:.1%}  fill_target={TARGET_FILL:.0%}')

In [ ]:
# ── Cell 6 · Build & stock warehouses A / B / C ───────────────────────────────
#
# A  Uniform random assignment (baseline)
# B  Load-minimising reorders  (same uniform initial stock as A)
# C  Load-maximising reorders  (same uniform initial stock as A)
#
# All three warehouses are built with the same seed so identical bin layouts;
# initial stocking also uses the same seed so the starting condition is
# identical.  Only reorder placement diverges across strategies.

def _build_warehouse(seed_layout, seed_stock, affinity=None):
    Aisle.next_aisle_id = 1
    random.seed(seed_layout)
    wh = Warehouse_Builder().from_config(warehouse_cfg).build()
    random.seed(seed_stock)
    mgr = Inventory_Manager(wh, affinity=affinity)
    return wh, mgr

def _overstock(manager, target=TARGET_FILL):
    total      = len(manager.warehouse.bins)
    target_n   = round(target * total)
    current    = len(manager.unavailable)
    if current >= target_n:
        return
    needed  = target_n - current
    weights = [c.demand.frequency for c in inventory.cartons]
    total_w = sum(weights)
    norm_w  = [w / total_w for w in weights]
    sample  = random.choices(inventory.cartons, weights=norm_w, k=needed * 3)
    manager.enqueue_all(sample, quantity=1)

t0 = time.perf_counter()

# Strategy A — uniform everywhere
_, manager_A = _build_warehouse(SEED_WORLD, SEED_WORLD + 100)
manager_A.enqueue_all(inventory.cartons, quantity=1)
_overstock(manager_A)

# Strategy B — uniform initial, load-minimising reorders
_, manager_B = _build_warehouse(SEED_WORLD, SEED_WORLD + 100, affinity=affinity_store)
manager_B.enqueue_all(inventory.cartons, quantity=1)
_overstock(manager_B)
manager_B.init_lift_state(affinity_store)
manager_B.assignment_fn = build_load_minimizing_assignment_fn(
    load_params, affinity_store, wp,
    manager_B._aisle_sku_sets, manager_B._aisle_lift_sum, manager_B._aisle_idx_sets,
)

# Strategy C — uniform initial, load-maximising reorders
_, manager_C = _build_warehouse(SEED_WORLD, SEED_WORLD + 100, affinity=affinity_store)
manager_C.enqueue_all(inventory.cartons, quantity=1)
_overstock(manager_C)
manager_C.init_lift_state(affinity_store)
manager_C.assignment_fn = build_load_maximizing_assignment_fn(
    load_params, affinity_store, wp,
    manager_C._aisle_sku_sets, manager_C._aisle_lift_sum, manager_C._aisle_idx_sets,
)

aisle_size_map     = {a.aisle_id: a.storage_size  for a in manager_A.warehouse.aisles}
aisle_unittype_map = {a.aisle_id: a.unit_type     for a in manager_A.warehouse.aisles}

fa = len(manager_A.unavailable) / total_bins
fb = len(manager_B.unavailable) / total_bins
fc = len(manager_C.unavailable) / total_bins
print(f'Stocked in {time.perf_counter()-t0:.1f}s')
print(f'A fill={fa:.1%}   B fill={fb:.1%}   C fill={fc:.1%}')

In [ ]:
# ── Cell 7 · Simulation loop ──────────────────────────────────────────────────
random.seed(SEED_BATCHES)

rec_A, rec_B, rec_C = [], [], []   # BatchStats lists
t_loop = time.perf_counter()
skipped = 0

for i in range(N_BATCHES):
    manager_A.check_reorders()
    manager_B.check_reorders()
    manager_C.check_reorders()

    batch = Batch(batch_cfg, inventory, affinity=affinity_store)
    ta    = Task.from_batch(batch, manager_A.warehouse, manager=manager_A)
    tb    = Task.from_batch(batch, manager_B.warehouse, manager=manager_B)
    tc    = Task.from_batch(batch, manager_C.warehouse, manager=manager_C)

    if not ta or not tb or not tc:
        skipped += 1
        continue

    ea = PickSimulation(ta, pick_cfg, manager=manager_A).run()
    eb = PickSimulation(tb, pick_cfg, manager=manager_B).run()
    ec = PickSimulation(tc, pick_cfg, manager=manager_C).run()

    rec_A.append(extract_batch_stats(ea, batch_id=i, k_pickers=K_PICKERS))
    rec_B.append(extract_batch_stats(eb, batch_id=i, k_pickers=K_PICKERS))
    rec_C.append(extract_batch_stats(ec, batch_id=i, k_pickers=K_PICKERS))

    if (i + 1) % 50 == 0:
        elapsed = time.perf_counter() - t_loop
        rate    = (i + 1) / elapsed
        dA = rec_A[-1].duration; dB = rec_B[-1].duration; dC = rec_C[-1].duration
        print(f'  Batch {i+1:4d}/{N_BATCHES}  A={dA:.0f}  B={dB:.0f}  C={dC:.0f}  '
              f'{rate:.2f} batches/s')

elapsed = time.perf_counter() - t_loop
n_done  = N_BATCHES - skipped
print(f'\nDone: {n_done} triplets in {elapsed:.1f}s  ({skipped} skipped)')

In [ ]:
# ── Cell 8 · Build DataFrames ─────────────────────────────────────────────────
def _bdf(recs, label):
    return pd.DataFrame([{
        'batch'          : r.batch_id,
        'strategy'       : label,
        'duration'       : r.duration,
        'num_tasks'      : r.num_tasks,
        'total_items'    : r.total_items,
        'throughput'     : r.total_items / r.duration if r.duration > 0 else 0,
        'concurrent'     : r.avg_concurrent_pickers,
        'picking_pct'    : r.picking_pct   * 100,
        'traveling_pct'  : r.traveling_pct * 100,
    } for r in recs])

df_A = _bdf(rec_A, 'A · Uniform')
df_B = _bdf(rec_B, 'B · Load-Min')
df_C = _bdf(rec_C, 'C · Load-Max')
df   = pd.concat([df_A, df_B, df_C], ignore_index=True)

display(df.groupby('strategy')[['duration','throughput','picking_pct','traveling_pct']]
          .agg(['mean','median','std']).round(2))

In [ ]:
# ── Cell 9 · Plot 1: Batch duration rolling mean ──────────────────────────────
_COL = {'A · Uniform': '#5b9bd5', 'B · Load-Min': '#f4a030', 'C · Load-Max': '#70ad47'}

fig, ax = plt.subplots(figsize=(12, 4))
for dfi in [df_A, df_B, df_C]:
    label = dfi['strategy'].iloc[0]
    roll  = dfi['duration'].rolling(PLOT_WINDOW, min_periods=1).mean()
    ax.plot(dfi['batch'], roll, label=label, color=_COL[label], lw=1.8)

ax.set_xlabel('Batch', fontsize=11)
ax.set_ylabel('Duration (rolling mean)', fontsize=11)
ax.set_title(f'Batch Completion Time — {PLOT_WINDOW}-batch rolling mean', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10 · Plot 2: Duration distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle('Batch Duration Distributions', fontsize=13, fontweight='bold')

def _kde_plot(ax, data, color, label):
    data = np.asarray(data, dtype=float)
    ax.hist(data, bins=30, color=color, alpha=0.6, edgecolor='white')
    if len(data) > 1 and data.max() > data.min():
        kde = gaussian_kde(data, bw_method='silverman')
        xs  = np.linspace(data.min(), data.max(), 300)
        ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 30,
                color=color, lw=2)
    ax.axvline(data.mean(),     color='red',    lw=1.5, ls='--',
               label=f'Mean {data.mean():.1f}')
    ax.axvline(np.median(data), color='orange', lw=1.5, ls=':',
               label=f'Median {np.median(data):.1f}')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Duration')
    ax.legend(fontsize=9)

for ax, dfi in zip(axes, [df_A, df_B, df_C]):
    label = dfi['strategy'].iloc[0]
    _kde_plot(ax, dfi['duration'].values, _COL[label], label)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11 · Plot 3: Picker time split ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Picker Time Split', fontsize=12, fontweight='bold')

labels  = [df['strategy'].iloc[0] for df in [df_A, df_B, df_C]]
colors  = [_COL[l] for l in labels]
pick_m  = [df['picking_pct'].mean()   for df in [df_A, df_B, df_C]]
trav_m  = [df['traveling_pct'].mean() for df in [df_A, df_B, df_C]]

x = np.arange(3)
axes[0].bar(x, pick_m,  color=colors, edgecolor='white', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_ylabel('%'); axes[0].set_title('Avg picking %')
axes[0].set_ylim(0, 100)

axes[1].bar(x, trav_m, color=colors, edgecolor='white', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_ylabel('%'); axes[1].set_title('Avg traveling %')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12 · Plot 4: Rolling throughput comparison ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
for dfi in [df_A, df_B, df_C]:
    label = dfi['strategy'].iloc[0]
    roll  = dfi['throughput'].rolling(PLOT_WINDOW, min_periods=1).mean()
    ax.plot(dfi['batch'], roll, label=label, color=_COL[label], lw=1.8)

ax.set_xlabel('Batch'); ax.set_ylabel('Items / time unit (rolling mean)')
ax.set_title('Throughput', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 13 · Summary delta table ─────────────────────────────────────────────
metrics = ['duration', 'throughput', 'picking_pct', 'traveling_pct']
rows = []
base = df_A[metrics].mean()
for dfi, name in [(df_B, 'B vs A'), (df_C, 'C vs A')]:
    m   = dfi[metrics].mean()
    row = {'comparison': name}
    for col in metrics:
        delta_pct = (m[col] - base[col]) / base[col] * 100 if base[col] != 0 else float('nan')
        row[col]        = round(m[col], 3)
        row[col + '_Δ%'] = round(delta_pct, 2)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('comparison')
print('\nMean values and % delta vs Uniform (A):')
display(summary)

In [ ]:
# ── Cell 14 · Load model sensitivity: scan λ and γ ───────────────────────────
# Re-runs a SHORT simulation for a grid of λ / γ values.
# Set SENSITIVITY_BATCHES small (20-50) for quick exploration.

SENSITIVITY_BATCHES = 30
LAMBDA_GRID = [0.5, 1.0, 1.5, 2.0]
GAMMA_GRID  = [1.0, 1.5, 2.0]

def _quick_run(lam, gam, n_batches, seed_b):
    """Run B vs A for given λ, γ; return (mean_dur_A, mean_dur_B, delta_pct)."""
    lp = LoadParams(lambda_=lam, k=LOAD_K, gamma=gam)

    Aisle.next_aisle_id = 1
    random.seed(SEED_WORLD)
    wh_a = Warehouse_Builder().from_config(warehouse_cfg).build()
    random.seed(SEED_WORLD + 100)
    mgr_a = Inventory_Manager(wh_a)
    mgr_a.enqueue_all(inventory.cartons, quantity=1)
    _overstock(mgr_a)

    Aisle.next_aisle_id = 1
    random.seed(SEED_WORLD)
    wh_b = Warehouse_Builder().from_config(warehouse_cfg).build()
    random.seed(SEED_WORLD + 100)
    mgr_b = Inventory_Manager(wh_b, affinity=affinity_store)
    mgr_b.enqueue_all(inventory.cartons, quantity=1)
    _overstock(mgr_b)
    mgr_b.init_lift_state(affinity_store)
    mgr_b.assignment_fn = build_load_minimizing_assignment_fn(
        lp, affinity_store, wp,
        mgr_b._aisle_sku_sets, mgr_b._aisle_lift_sum, mgr_b._aisle_idx_sets,
    )

    random.seed(seed_b)
    durs_a, durs_b = [], []
    for i in range(n_batches):
        mgr_a.check_reorders(); mgr_b.check_reorders()
        batch = Batch(batch_cfg, inventory, affinity=affinity_store)
        ta = Task.from_batch(batch, wh_a, manager=mgr_a)
        tb = Task.from_batch(batch, wh_b, manager=mgr_b)
        if not ta or not tb:
            continue
        ea = PickSimulation(ta, pick_cfg, manager=mgr_a).run()
        eb = PickSimulation(tb, pick_cfg, manager=mgr_b).run()
        bsa = extract_batch_stats(ea, i, K_PICKERS)
        bsb = extract_batch_stats(eb, i, K_PICKERS)
        durs_a.append(bsa.duration)
        durs_b.append(bsb.duration)

    ma, mb = np.mean(durs_a), np.mean(durs_b)
    delta  = (mb - ma) / ma * 100 if ma > 0 else float('nan')
    return ma, mb, delta

print(f'Sensitivity scan: {len(LAMBDA_GRID)}×{len(GAMMA_GRID)} grid, '
      f'{SENSITIVITY_BATCHES} batches each…')
t0     = time.perf_counter()
grid   = []
for lam in LAMBDA_GRID:
    for gam in GAMMA_GRID:
        ma, mb, delta = _quick_run(lam, gam, SENSITIVITY_BATCHES, SEED_BATCHES)
        grid.append({'λ': lam, 'γ': gam, 'dur_A': round(ma,1),
                     'dur_B': round(mb,1), 'Δ% (B-A)': round(delta,2)})

df_grid = pd.DataFrame(grid)
print(f'Done in {time.perf_counter()-t0:.1f}s')
display(df_grid.pivot(index='λ', columns='γ', values='Δ% (B-A)'))

In [ ]:
# ── Cell 15 · Plot 5: Sensitivity heatmap ─────────────────────────────────────
pivot = df_grid.pivot(index='λ', columns='γ', values='Δ% (B-A)').astype(float)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto',
               vmin=min(-5, pivot.values.min()),
               vmax=max( 5, pivot.values.max()))
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'γ={v}' for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'λ={v}' for v in pivot.index])
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i,j]:.1f}%', ha='center', va='center',
                fontsize=10, color='black')
plt.colorbar(im, ax=ax, label='Δ% duration (B − A)')
ax.set_title('Load-Min B vs Uniform A — Δ% batch duration\n(negative = B faster)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 16 · Queue depth over time ───────────────────────────────────────────
# Re-run a short simulation tracking queue depth after each check_reorders call.

QUEUE_BATCHES = min(N_BATCHES, 100)
random.seed(SEED_BATCHES)

Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)
wh_q = Warehouse_Builder().from_config(warehouse_cfg).build()
random.seed(SEED_WORLD + 100)
mgr_q = Inventory_Manager(wh_q, affinity=affinity_store)
mgr_q.enqueue_all(inventory.cartons, quantity=1)
_overstock(mgr_q)
mgr_q.init_lift_state(affinity_store)
mgr_q.assignment_fn = build_load_minimizing_assignment_fn(
    load_params, affinity_store, wp,
    mgr_q._aisle_sku_sets, mgr_q._aisle_lift_sum, mgr_q._aisle_idx_sets,
)

queue_depths, reorder_counts = [], []
random.seed(SEED_BATCHES)
for i in range(QUEUE_BATCHES):
    triggered = mgr_q.check_reorders()
    queue_depths.append(mgr_q.queue_depth)
    reorder_counts.append(len(triggered))
    batch = Batch(batch_cfg, inventory, affinity=affinity_store)
    tasks = Task.from_batch(batch, wh_q, manager=mgr_q)
    if tasks:
        PickSimulation(tasks, pick_cfg, manager=mgr_q).run()

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(queue_depths, color='#e05c40', lw=1.5)
axes[0].set_ylabel('Unplaced units in queue'); axes[0].grid(alpha=0.3)
axes[0].set_title('Reorder queue depth over batches (Strategy B)', fontsize=11)
axes[1].bar(range(QUEUE_BATCHES), reorder_counts, color='#5b9bd5', alpha=0.7)
axes[1].set_ylabel('New SKUs triggered'); axes[1].set_xlabel('Batch'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()